# Neural Networks from Scratch — 1. NumPy Foundations

**Notebook 1 of 4**

| # | Notebook | What you build |
|---|---|---|
| **1** | **NumPy foundations** | **A single neuron, trained by hand** |
| 2 | A network from scratch | A 2-layer network with backpropagation |
| 3 | The same network in PyTorch | Tensors, autograd, `nn.Module` |
| 4 | MNIST and a small CNN | A real image classifier |

---

## What this notebook is for

By the end of this notebook you will have written a *learning* program: a single
artificial neuron that starts out useless, looks at some data, and gradually
improves until it can classify points correctly.

There is no magic in a neural network. There is multiplication, addition, a
squashing function, and a rule for nudging numbers in the right direction. We are
going to build exactly those four things, in that order.

## How to use it

Run the cells in order, top to bottom (`Shift+Enter`). Don't just read — **change
numbers and re-run**. Most cells end with a suggestion for something to try. The
suggestions are the actual lesson; the text is just scaffolding.

Cells marked **🔬 Experiment** are meant to be broken on purpose.

## What you need to know already

Basic Python: variables, functions, `for` loops, lists. That's it.

We will need a little maths vocabulary — *slope*, *steepness*, *downhill* — but we
will keep it intuitive and look at pictures instead of deriving formulas. Where a
formula appears it is always followed by the code that does the same thing, and the
code is the part that matters.

---
# Part 0 — Setup

You need three packages: `numpy`, `matplotlib` and `scikit-learn`.

If you are missing any of them, run this in a terminal (or in a cell with a `!` in
front):

```
pip install numpy matplotlib scikit-learn
```

Everything in notebooks 1 and 2 runs on a laptop CPU in seconds. No GPU needed,
here or later.

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn

print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

numpy       2.5.3
matplotlib  3.11.2
scikit-learn 1.9.1


### A note on the plotting code

Every picture in these notebooks is made with **matplotlib**. Matplotlib is not the
point of the course, so here is the 30-second version — enough that the plotting
cells don't look like noise:

- `plt.figure(figsize=(w, h))` — start a new picture, `w` by `h` inches.
- `plt.plot(x, y)` — draw a line through the points.
- `plt.scatter(x, y, c=colors)` — draw dots, optionally coloured by a third array.
- `plt.contourf(...)` — fill regions of a 2D grid with colour. We use this to shade
  what a model predicts everywhere on the plane.
- `plt.xlabel / ylabel / title / legend / show` — labels and display.

When a plot needs more than a few lines, we wrap it in a helper function and you can
happily treat that helper as a black box. The first time we write one, we'll walk
through it.

In [2]:
# Plot styling - cosmetic only, safe to ignore.
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 110

# A fixed random generator, so your numbers match the text.
# Change the seed to see different random draws.
rng = np.random.default_rng(seed=0)

---
# Part 1 — NumPy essentials for machine learning

NumPy is the reason Python is usable for machine learning. It gives you the
**array**: a grid of numbers that you can do arithmetic on all at once.

You only need a handful of NumPy ideas for everything that follows:

1. arrays and their **shape**
2. **vectorized** arithmetic (no Python loops)
3. **matrix multiplication**, `@`
4. **broadcasting**

Let's take them one at a time.

## 1.1 Arrays and shapes

An array is a grid of numbers. Its `.shape` tells you how big the grid is, as a
tuple: `(rows, columns)`.

**Shape mistakes are the single most common bug in machine learning code.**

In [3]:
a = np.array([1.0, 2.0, 3.0])  # 1D: a vector of 3 numbers
b = np.array(
    [
        [1.0, 2.0, 3.0],  # 2D: 2 rows, 3 columns
        [4.0, 5.0, 6.0],
    ]
)

print("a =", a)
print("a.shape =", a.shape)  # (3,)   -> 1D, length 3
print()
print("b =")
print(b)
print("b.shape =", b.shape)  # (2, 3) -> 2 rows, 3 columns

a = [1. 2. 3.]
a.shape = (3,)

b =
[[1. 2. 3.]
 [4. 5. 6.]]
b.shape = (2, 3)


Read `(2, 3)` as "2 rows, 3 columns".

Note the comma in `(3,)`. That is Python's way of writing a one-element tuple, and it
marks `a` as **one-dimensional**. A 1D array of length 3 is *not* the same thing as a
2D array of shape `(1, 3)` or `(3, 1)`, and confusing them causes a lot of grief.
Let's make the difference visible.

In [4]:
row = np.array([[1.0, 2.0, 3.0]])          # note the double brackets
col = np.array([[1.0], [2.0], [3.0]])

print("a   shape", a.shape,   "  <- 1D, no orientation")
print("row shape", row.shape, "<- 2D, one row")
print("col shape", col.shape, "<- 2D, one column")

# reshape(-1, 1) is the idiom for "make this a single column".
# The -1 means "work the size out for me".
print()
print("a.reshape(-1, 1) shape:", a.reshape(-1, 1).shape)

a   shape (3,)   <- 1D, no orientation
row shape (1, 3) <- 2D, one row
col shape (3, 1) <- 2D, one column

a.reshape(-1, 1) shape: (3, 1)


### Our data layout convention

Throughout all four notebooks we store data as a 2D array with:

- **one row per example** (one flower, one image, one customer)
- **one column per feature** (petal length, pixel brightness, age)

So `X.shape == (n_samples, n_features)`. This is the standard convention in
scikit-learn and PyTorch alike, and sticking to it prevents most shape confusion.

In [ ]:
# 5 examples, 2 features each.
X = rng.normal(size=(5, 2))
print(X)
print("X.shape =", X.shape, "-> 5 examples, 2 features")

## 1.2 Vectorized arithmetic

Arithmetic on arrays happens **element by element**, across the whole array at once.
No loop required. This matters for two reasons: the code is shorter, and it runs
one or two orders of magnitude faster, because the loop happens in compiled C rather
than in Python.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])

print("x + 10  =", x + 10)        # adds 10 to every element
print("x * 2   =", x * 2)
print("x ** 2  =", x ** 2)
print("x > 2.5 =", x > 2.5)       # elementwise comparison -> array of True/False
print()
print("np.exp(x)  =", np.exp(x))  # e to the power of each element
print("np.mean(x) =", np.mean(x)) # collapses the array to one number
print("np.sum(x)  =", np.sum(x))

Here is the same computation written both ways, so you can see the speed difference
for yourself. One million numbers, squared.

In [ ]:
import time

big = rng.normal(size=1_000_000)

t0 = time.perf_counter()
result_loop = [v ** 2 for v in big]           # the Python way
t1 = time.perf_counter()
result_vec = big ** 2                         # the NumPy way
t2 = time.perf_counter()

print(f"Python loop : {t1 - t0:.4f} s")
print(f"NumPy       : {t2 - t1:.4f} s")
print(f"speed-up    : {(t1 - t0) / (t2 - t1):.0f}x")

**Rule of thumb:** if you are writing a `for` loop over data points in NumPy code,
there is almost always a vectorized way to do it instead. The one loop we *will*
keep is the loop over training steps, because each step genuinely depends on the one
before it.

## 1.3 Matrix multiplication — the `@` operator

This is the operation at the heart of every neural network, so it is worth a minute.

Matrix multiplication combines two arrays by taking **rows of the first against
columns of the second**. For each output entry, you multiply the matching numbers
pairwise and add them up.

The shape rule is the only thing you have to remember:

```
(n, k) @ (k, m)  ->  (n, m)
      ^^^^^^^  these must match; they cancel out
```

The inner numbers must be equal, and they disappear from the result.

In [ ]:
A_ = np.array([[1.0, 2.0],
               [3.0, 4.0],
               [5.0, 6.0]])      # (3, 2)

B_ = np.array([[10.0, 20.0, 30.0, 40.0],
               [50.0, 60.0, 70.0, 80.0]])   # (2, 4)

C_ = A_ @ B_

print("A_ shape", A_.shape)
print("B_ shape", B_.shape)
print("C_ shape", C_.shape, " <- (3,2) @ (2,4) -> (3,4)")
print()
print(C_)

# Check one entry by hand: top-left = row 0 of A_ against column 0 of B_
print()
print("C_[0, 0] by hand:", 1.0 * 10.0 + 2.0 * 50.0, " matches", C_[0, 0])

**🔬 Experiment.** Uncomment the line below and run it. Read the error message
carefully — you will see this one a lot, and learning to parse it now saves you hours
later.

In [ ]:
# A_ @ A_       # (3,2) @ (3,2)  ->  2 and 3 don't match  ->  ValueError

# The error says something like:
#   matmul: Input operand 1 has a mismatch in its core dimension 0 ...
# Translation: "the inner dimensions don't line up". Print both shapes and see
# which number needs to move.

### Why this is the core of a neural network

A **layer** of a neural network takes a batch of examples and produces a batch of
outputs. With our row-per-example layout that is exactly one matrix multiply:

```
X @ W   ->   (n_samples, n_features) @ (n_features, n_units)  ->  (n_samples, n_units)
```

Every example in the batch gets processed in a single operation. That is the whole
trick. When you hear that neural networks are "just matrix multiplications", this is
what is meant — and after notebook 2 you will have written one yourself.

In [ ]:
X = rng.normal(size=(5, 2))    # 5 examples, 2 features
W = rng.normal(size=(2, 3))    # 2 features in, 3 units out

Z = X @ W
print("X", X.shape, "@ W", W.shape, "-> Z", Z.shape)

## 1.4 Broadcasting

Broadcasting is NumPy stretching a smaller array to fit a bigger one, automatically,
without copying any memory.

We need it because of the **bias**: every unit in a layer has one bias number, and it
has to be added to that unit's output for *every* example in the batch. Broadcasting
does that for us.

In [ ]:
Z = rng.normal(size=(5, 3))    # 5 examples, 3 units
b = np.array([100.0, 200.0, 300.0])   # one bias per unit, shape (3,)

print("Z shape", Z.shape, "+ b shape", b.shape, "->", (Z + b).shape)
print()
print(np.round(Z + b, 2))

Look at the result: every value in the first column got +100, the second +200, the
third +300. The bias row was stretched down across all 5 examples.

**The broadcasting rule.** Line the shapes up from the *right*. Two dimensions are
compatible if they are equal, or if one of them is 1 (or missing).

```
Z      (5, 3)
b         (3,)     ->  treated as (1, 3)  ->  stretched to (5, 3)   ✓

Z      (5, 3)
bad       (5,)     ->  treated as (1, 5)  ->  3 vs 5, no match      ✗
```

In [ ]:
# The classic broadcasting trap: an array that is "the right length" but the
# wrong orientation.
try:
    Z + np.array([1.0, 2.0, 3.0, 4.0, 5.0])     # shape (5,) vs (5, 3)
except ValueError as e:
    print("ValueError:", e)

# Fix: say explicitly that it is a column, one value per example.
print()
print("as a column:", (Z + np.array([1.0, 2.0, 3.0, 4.0, 5.0]).reshape(-1, 1)).shape)

> **Debugging habit.** When NumPy code misbehaves, print the shape of every array
> involved before you do anything else. Nine times out of ten the bug is right there,
> and you've just saved yourself half an hour of staring at the maths.

That's all the NumPy you need. On to the actual neural network.

---
# Part 2 — A single neuron

A neuron is a small function. It takes some numbers in, and puts one number out. It
does this in two steps:

**Step 1 — weighted sum.** Multiply each input by a *weight*, add them up, then add a
*bias*:

$$z = x_1 w_1 + x_2 w_2 + \dots + b$$

The weights say how much each input matters. A large positive weight means "this
input pushes the answer up"; a large negative weight means "this input pushes it
down"; a weight near zero means "ignore this input". The bias shifts the whole thing
up or down regardless of the inputs.

**Step 2 — activation.** Pass `z` through a squashing function `a = f(z)`.

The weights and the bias are the **parameters** — the numbers that get *learned*.
Everything else about the network is fixed by us in advance.

In [ ]:
def neuron(x, w, b):
    """One neuron, one example.

    x: inputs,  shape (n_features,)
    w: weights, shape (n_features,)
    b: bias,    a single number

    Returns the weighted sum z (we'll add the activation in a moment).
    """
    return x @ w + b


x = np.array([2.0, 3.0])       # one example with two features
w = np.array([0.5, -1.0])      # its two weights
b = 1.0

print("z =", neuron(x, w, b))
print("by hand:", 2.0 * 0.5 + 3.0 * -1.0 + 1.0)

**🔬 Try it.** Before running the cell below, predict what happens:

- set `w = [10.0, 0.0]` — the neuron now ignores the second feature entirely
- set `b = -100` — the output is dragged far negative whatever the inputs are

Change the numbers and check your prediction.

In [ ]:
for w_try, b_try in [(np.array([0.5, -1.0]), 1.0),
                     (np.array([10.0, 0.0]), 1.0),
                     (np.array([0.5, -1.0]), -100.0)]:
    print(f"w={w_try}, b={b_try:7.1f}  ->  z = {neuron(x, w_try, b_try):8.2f}")

## 2.1 Activation functions

Why not stop at the weighted sum? Because a weighted sum is a *straight line*, and
stacking straight lines on top of straight lines only ever gives you another straight
line. Without a non-linear activation function, a hundred-layer network can do no
more than a single layer can. (We will demonstrate this dramatically in notebook 2.)

The activation function bends the line. Two of them cover most of what you'll meet:

**Sigmoid** squashes anything into the range 0 to 1, which makes it readable as a
probability. Large positive input → near 1. Large negative → near 0. Zero → exactly
0.5.

**ReLU** ("rectified linear unit") is almost embarrassingly simple: negatives become
zero, positives pass through unchanged. It is the default inside modern networks
because it is fast and trains well.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    return np.maximum(0.0, z)


zs = np.linspace(-6, 6, 200)      # 200 evenly spaced values from -6 to 6

# Two plots side by side: plt.subplots(1, 2) makes a 1-row, 2-column grid and
# hands back the figure plus an array of axes. ax.plot works like plt.plot.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(zs, sigmoid(zs), linewidth=2)
axes[0].axhline(0.5, linestyle="--", linewidth=1, color="gray")
axes[0].axvline(0.0, linestyle="--", linewidth=1, color="gray")
axes[0].set_title("sigmoid: squashes into (0, 1)")
axes[0].set_xlabel("z"); axes[0].set_ylabel("f(z)")

axes[1].plot(zs, relu(zs), linewidth=2, color="darkorange")
axes[1].set_title("ReLU: max(0, z)")
axes[1].set_xlabel("z"); axes[1].set_ylabel("f(z)")

plt.tight_layout()
plt.show()

In [ ]:
# Sigmoid on some specific values, to build intuition for the numbers.
for z in [-10, -2, -1, 0, 1, 2, 10]:
    print(f"sigmoid({z:4}) = {sigmoid(z):.4f}")

Notice how flat sigmoid is at the ends: `sigmoid(-10)` and `sigmoid(-2)` are both
"basically zero". That flatness will come back to bite us later — it is why very deep
networks built from sigmoids are hard to train, and part of why ReLU took over.

## 2.2 A neuron is a line

Here's the geometric picture, and it's worth internalising because it explains what
a single neuron can and cannot do.

With two input features, the neuron computes `z = x1*w1 + x2*w2 + b`. The set of
points where `z = 0` is a **straight line** on the plane. On one side `z` is
positive, on the other it's negative. Push `z` through sigmoid and you get: "above
0.5 on one side, below 0.5 on the other".

So **one neuron splits the plane with a straight line**. The weights control the
line's angle; the bias slides it back and forth.

Let's see that. First we need some data.

In [ ]:
from sklearn.datasets import make_blobs

# Two clearly separated clusters, 2 features each.
X, y = make_blobs(n_samples=200, centers=2, n_features=2,
                  cluster_std=1.2, random_state=4)

print("X", X.shape, " y", y.shape)
print("labels present:", np.unique(y))

plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.title("Two classes, nearly separable by a straight line")
plt.show()

### A reusable plotting helper

We are going to want "shade the plane by what the model predicts" many times, in
this notebook and the next two. Here is the helper, with the logic spelled out
once. After this, feel free to treat it as a black box.

The idea: lay a fine grid of points over the whole plot area, ask the model for a
prediction at every grid point, and colour the grid by the answer. The boundary
between the colours is the model's decision boundary.

In [ ]:
def plot_decision_boundary(predict_fn, X, y, title="", ax=None):
    """Shade the plane by what predict_fn says, then scatter the real data on top.

    predict_fn: takes an array of shape (n_points, 2), returns a value per point
                (we'll pass in probabilities between 0 and 1)
    X, y:       the data to overlay
    """
    if ax is None:
        _, ax = plt.subplots()

    # 1. Work out the area to cover: the data's range, plus a small margin.
    pad = 0.5
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad

    # 2. Build a grid of points covering that area.
    #    meshgrid turns two 1D ranges into two 2D coordinate arrays.
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    # 3. Flatten the grid into a (n_points, 2) array - our usual data layout -
    #    so the model can score every point in one vectorized call.
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_fn(grid).reshape(xx.shape)     # reshape back into grid form

    # 4. Shade the grid, and draw a solid line at the 0.5 boundary.
    ax.contourf(xx, yy, zz, levels=50, cmap="coolwarm", alpha=0.7)
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)

    # 5. The real data points on top.
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
    ax.set_title(title)
    ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
    return ax

Now let's plug in a neuron with weights we choose *by hand* — no learning yet. Just
to confirm that a neuron does indeed draw a line, and that the weights control where
it goes.

In [ ]:
def neuron_predict(grid, w, b):
    """Run a batch of points through the neuron. grid: (n_points, 2)."""
    return sigmoid(grid @ w + b)


fig, axes = plt.subplots(1, 3, figsize=(14, 4))

settings = [(np.array([1.0, 0.0]),  0.0, "w=[1, 0], b=0"),
            (np.array([0.0, 1.0]),  0.0, "w=[0, 1], b=0"),
            (np.array([1.0, 1.0]), -2.0, "w=[1, 1], b=-2")]

for ax, (w_try, b_try, label) in zip(axes, settings):
    plot_decision_boundary(lambda g: neuron_predict(g, w_try, b_try),
                           X, y, title=label, ax=ax)

plt.tight_layout()
plt.show()

Three different lines, none of them any good. The weights control the line's angle,
the bias slides it around, and right now we are just guessing.

**🔬 Try it.** Go back and edit `settings` until you find weights that actually
separate the two clusters. It is fiddly, and that fiddliness is the point: we want
the computer to do this search for us.

That search is called **training**, and it is the subject of Part 3.

---
# Part 3 — Learning by gradient descent

To make the computer find good weights, we need two things:

1. a **loss** — a single number that says how wrong the current weights are
2. a **rule** for changing the weights to make that number smaller

Then we repeat step 2 until the loss stops going down. That's training. All of it.

## 3.1 The loss function

A loss takes the model's predictions and the true answers, and returns one number.
Low is good.

**Mean squared error (MSE)** — average of the squared differences. The natural choice
when you're predicting a quantity (a price, a temperature).

**Binary cross-entropy (BCE)** — the right choice when you're predicting a
probability, which is our case. It is much harsher than MSE about confident mistakes:
predicting 0.99 when the answer is 0 is penalised enormously. That harshness is
exactly what you want; it keeps the model from getting confidently wrong, and it
makes training better behaved.

In [ ]:
def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)


def bce_loss(y_pred, y_true):
    # Clip away from exactly 0 and 1, otherwise log(0) = -infinity and
    # everything downstream becomes NaN. A standard, necessary bit of hygiene.
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


# How each loss reacts as a prediction gets more wrong (true answer is 1).
preds = np.linspace(0.001, 0.999, 300)
truth = np.ones_like(preds)

plt.figure()
plt.plot(preds, [(p - 1.0) ** 2 for p in preds], label="MSE", linewidth=2)
plt.plot(preds, -np.log(preds), label="binary cross-entropy", linewidth=2)
plt.xlabel("predicted probability (true answer = 1)")
plt.ylabel("loss")
plt.title("Cross-entropy punishes confident mistakes far harder")
plt.legend()
plt.show()

In [ ]:
y_true = np.array([1.0, 1.0, 0.0, 0.0])

for name, y_pred in [("nearly right    ", np.array([0.9, 0.8, 0.2, 0.1])),
                     ("unsure          ", np.array([0.5, 0.5, 0.5, 0.5])),
                     ("confidently wrong", np.array([0.05, 0.1, 0.9, 0.95]))]:
    print(f"{name}  MSE {mse_loss(y_pred, y_true):.3f}   BCE {bce_loss(y_pred, y_true):.3f}")

## 3.2 Gradient descent, in one dimension

Now the learning rule. Here is the whole idea, stripped of everything else.

Imagine you are standing somewhere on a hillside in thick fog. You want to reach the
bottom of the valley, but you can only see the ground at your feet. What do you do?
You feel which way is downhill, take a small step that way, and repeat.

That is gradient descent. The **gradient** is the "which way is downhill and how
steep" measurement. The **learning rate** is the size of your step.

```
new_value = old_value  -  learning_rate * gradient
```

The minus sign is there because the gradient points *uphill*, and we want to go the
other way.

Let's watch it work on a simple bowl-shaped function, `f(x) = x²`, whose gradient is
`2x`. Here we know the gradient formula, so there is no mystery — the mystery is
saved for notebook 2, where we work it out for a whole network.

In [ ]:
def f(x):
    return x ** 2


def grad_f(x):
    return 2 * x      # the slope of x^2 at the point x


x = 8.0               # our starting point, up the side of the bowl
learning_rate = 0.1
history = [x]

for step in range(25):
    g = grad_f(x)                       # which way is uphill, and how steeply
    x = x - learning_rate * g           # step the other way
    history.append(x)

print(f"start 8.000  ->  after 25 steps {x:.6f}   (the true minimum is 0)")
print("first few values:", np.round(history[:6], 3))

In [ ]:
# Draw the bowl, and the path our steps took down it.
xs = np.linspace(-9, 9, 200)
history = np.array(history)

plt.figure(figsize=(7, 4.5))
plt.plot(xs, f(xs), linewidth=2, label="f(x) = x²")
plt.plot(history, f(history), "o-", color="crimson", markersize=5,
         linewidth=1, label="gradient descent path")
plt.scatter([history[0]], [f(history[0])], color="black", zorder=5, s=80,
            label="start")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("Rolling downhill, one step at a time")
plt.legend()
plt.show()

Look at the spacing of the dots: big steps where the hill is steep, small steps as it
flattens out. Nobody programmed that. It falls out of the fact that the gradient
*is* the steepness — so the same rule automatically takes bold steps when it is far
from the answer and cautious ones when it is close.

## 3.3 🔬 The learning rate

The learning rate is the one number you will spend the most time tuning in your ML
career. Too small and training takes forever; too large and it overshoots and blows
up. Let's see all three regimes at once.

In [ ]:
def descend(lr, steps=25, start=8.0):
    x, hist = start, [start]
    for _ in range(steps):
        x = x - lr * grad_f(x)
        hist.append(x)
    return np.array(hist)


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xs = np.linspace(-11, 11, 200)

for ax, (lr, label) in zip(axes, [(0.01, "too small (0.01)"),
                                  (0.10, "about right (0.1)"),
                                  (1.01, "too large (1.01)")]):
    h = descend(lr)
    ax.plot(xs, f(xs), linewidth=2)
    ax.plot(h, f(h), "o-", color="crimson", markersize=4, linewidth=1)
    ax.set_title(f"{label}\nafter 25 steps: x = {h[-1]:.3f}")
    ax.set_xlabel("x")

plt.tight_layout()
plt.show()

- **Too small:** barely moved. It *would* get there, with enough patience.
- **About right:** converged in a handful of steps.
- **Too large:** each step overshoots the bottom and lands *further up* the other
  side. The path spirals outward and the numbers explode.

That third picture is what "my loss became NaN" looks like from the inside. When
training blows up, the learning rate is the first thing to turn down.

**🔬 Try it.** Change `1.01` to `0.99`. It still oscillates from side to side, but
now it slowly shrinks inward instead of exploding. The boundary between working and
diverging is often razor-thin.

## 3.4 Training a real neuron

Time to put it together. We will train the neuron from Part 2 on the two blobs, so
it finds a good separating line by itself.

One new ingredient: we need the gradient of the *loss* with respect to `w` and `b`.
We promised to keep the maths light, so here is the result and the intuition, and we
will take the derivation properly in notebook 2.

For a sigmoid neuron with binary cross-entropy loss, the gradients work out to be
remarkably clean:

$$\text{error} = \hat{y} - y \qquad
\frac{\partial L}{\partial w} = \frac{1}{n}\,X^{T}\,\text{error} \qquad
\frac{\partial L}{\partial b} = \text{mean}(\text{error})$$

In words: **the error is just "prediction minus truth"**, and the gradient for each
weight is that error weighted by the input it was attached to. If a feature was large
and the prediction was too high, that feature's weight gets pulled down hard. If the
feature was zero, it wasn't to blame, and its weight doesn't move. That is the whole
sentiment of backpropagation, and you now have it.

(The sigmoid and the cross-entropy each contribute an ugly term, and those terms
cancel each other exactly. A happy accident, and another reason this pairing is
standard.)

In [ ]:
def train_neuron(X, y, learning_rate=0.1, n_steps=200):
    """Train one sigmoid neuron with gradient descent. Returns w, b, loss history."""
    n_samples, n_features = X.shape

    # Start from small random weights. Zeros would work for a single neuron,
    # but random is the habit we want for real networks (see notebook 2).
    rng_local = np.random.default_rng(0)
    w = rng_local.normal(size=n_features) * 0.01
    b = 0.0

    losses = []
    for step in range(n_steps):
        # ---- forward pass: what does the model currently predict? ----
        z = X @ w + b                 # (n_samples,)
        y_pred = sigmoid(z)           # (n_samples,) probabilities
        losses.append(bce_loss(y_pred, y))

        # ---- backward pass: which way is downhill? ----
        error = y_pred - y            # (n_samples,)
        grad_w = (X.T @ error) / n_samples      # (n_features,)
        grad_b = np.mean(error)                 # scalar

        # ---- update: take a step downhill ----
        w = w - learning_rate * grad_w
        b = b - learning_rate * grad_b

    return w, b, np.array(losses)


y_f = y.astype(float)     # labels as floats, to keep the arithmetic clean
w_trained, b_trained, losses = train_neuron(X, y_f, learning_rate=0.5, n_steps=300)

print("learned w =", np.round(w_trained, 3))
print("learned b =", round(b_trained, 3))
print(f"loss: {losses[0]:.4f} at the start  ->  {losses[-1]:.4f} at the end")

Read that training loop again — it is only about ten lines, and **every neural
network you will ever train has exactly this shape**:

1. **forward** — run the data through the model to get predictions
2. **loss** — measure how wrong they are
3. **backward** — compute the gradients
4. **update** — nudge the parameters downhill
5. repeat

Notebook 2 makes step 3 harder. Notebook 3 hands step 3 to PyTorch. The loop itself
never changes.

In [ ]:
# The loss curve: the single most useful diagnostic plot in machine learning.
plt.figure()
plt.plot(losses, linewidth=2)
plt.xlabel("training step"); plt.ylabel("binary cross-entropy loss")
plt.title("The loss going down is what 'learning' looks like")
plt.show()

In [ ]:
# And the boundary it found - compare with our hand-picked guesses in Part 2.
preds = sigmoid(X @ w_trained + b_trained)
accuracy = np.mean((preds > 0.5) == y)

plot_decision_boundary(lambda g: sigmoid(g @ w_trained + b_trained),
                       X, y, title=f"Learned boundary - accuracy {accuracy:.1%}")
plt.show()

It found the line. Nobody told it where to put it; it felt its way downhill 300 times
and arrived.

By the way, what we just built has a name. A single sigmoid neuron trained with
cross-entropy *is* **logistic regression** — a workhorse of statistics since the
1950s. You have not built a toy version of it; you have built the real thing. A
neural network is what you get when you wire a lot of these together, which is
notebook 2.

## 3.5 🔬 Experiments

Three things to try. Run the cell, then change the numbers yourself.

In [ ]:
# Experiment 1: learning rate, on the real problem this time.
plt.figure(figsize=(7, 4.5))
for lr in [0.001, 0.01, 0.1, 1.0, 10.0]:
    _, _, l = train_neuron(X, y_f, learning_rate=lr, n_steps=300)
    plt.plot(l, label=f"lr = {lr}", linewidth=2)

plt.xlabel("training step"); plt.ylabel("loss")
plt.title("Same model, same data, only the learning rate differs")
plt.legend()
plt.show()

`lr = 0.001` has barely started after 300 steps. `lr = 10.0` is jumpy — it lurches
about instead of settling. Somewhere in the middle is the sweet spot, and finding it
is usually a matter of trying a few values an order of magnitude apart, exactly as
we've done here.

In [ ]:
# Experiment 2: how long do we need to train?
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, n in zip(axes, [1, 10, 50, 300]):
    w_n, b_n, _ = train_neuron(X, y_f, learning_rate=0.5, n_steps=n)
    acc = np.mean((sigmoid(X @ w_n + b_n) > 0.5) == y)
    plot_decision_boundary(lambda g, w_=w_n, b_=b_n: sigmoid(g @ w_ + b_),
                           X, y, title=f"{n} steps - {acc:.0%}", ax=ax)

plt.tight_layout()
plt.show()

Watch the boundary swing into place. After a single step it is still essentially the
random line it started with; by 50 it is nearly right.

(The `w_=w_n, b_=b_n` in that lambda is a Python detail rather than an ML one: it
captures the *current* values of `w_n` and `b_n` as default arguments. Without it,
all four lambdas would end up sharing the final values from the loop — a classic
Python gotcha called late binding.)

In [ ]:
# Experiment 3: make the problem harder by overlapping the clusters.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, spread in [(axes[0], 1.0), (axes[1], 2.5), (axes[2], 4.5)]:
    Xh, yh = make_blobs(n_samples=200, centers=2, n_features=2,
                        cluster_std=spread, random_state=4)
    yh_f = yh.astype(float)
    w_h, b_h, _ = train_neuron(Xh, yh_f, learning_rate=0.5, n_steps=300)
    acc = np.mean((sigmoid(Xh @ w_h + b_h) > 0.5) == yh)
    plot_decision_boundary(lambda g, w_=w_h, b_=b_h: sigmoid(g @ w_ + b_),
                           Xh, yh, title=f"cluster_std={spread} - {acc:.0%}", ax=ax)

plt.tight_layout()
plt.show()

As the clusters blur into each other, accuracy falls — and *no* straight line could
do better. The model is not failing; the problem genuinely has no clean answer. Being
able to tell those two cases apart is a real skill, and it saves a lot of wasted
effort chasing accuracy that isn't there.

But some problems *do* have a clean answer that just isn't a straight line. That's
where one neuron runs out of road, and where notebook 2 begins.

---
# Recap

You now have, in working code:

- **NumPy for ML** — shapes, vectorization, `@`, broadcasting, and the habit of
  printing `.shape` when things go wrong
- **A neuron** — weighted sum plus activation; one neuron draws one straight line
- **A loss** — one number for "how wrong are we"; cross-entropy for probabilities
- **Gradient descent** — feel downhill, take a step, repeat; the learning rate is the
  step size and it matters enormously
- **A training loop** — forward, loss, backward, update. The same five lines will
  still be there in notebook 4, running on a convolutional network.

### Check yourself

Without scrolling up, can you answer these?

1. `X` has shape `(100, 5)` and `W` has shape `(5, 3)`. What shape is `X @ W`? What
   do the numbers 100, 5 and 3 mean in ML terms?
2. Why does a network need a non-linear activation function at all?
3. Your loss goes up and up until it becomes `NaN`. What is the first thing you'd
   change?
4. A single neuron gets 70% accuracy on your data. Name two very different
   explanations for that.

### Optional stretch

- Add a third feature to the blobs (`n_features=3`) and confirm the training code
  works unchanged. Nothing in `train_neuron` assumed there were two.
- Swap `bce_loss` for `mse_loss` in the training loop and compare the loss curves.
  (You'd need the matching gradient for a fair fight — but try it anyway and see.)
- Write `predict(X, w, b)` that returns hard 0/1 labels rather than probabilities.

### Next

**Notebook 2 — A neural network from scratch.** We meet a dataset that no straight
line can solve, add a hidden layer to fix it, and derive backpropagation properly.
It is the most demanding of the four notebooks and the one that makes everything
afterwards feel easy.